<a href="https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/revisados/06bis_AjusteNoLineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 06bis — Ajuste no lineal

Ejemplo de imagen de radar de apertura sintética (SAR) de polarización simple, en escala de grises, solo a modo ilustrativo del tipo de dato del que puede venir la distribución de ruido que vamos a ajustar en este notebook:

![Imagen de radar Cassini (SAR) del cráter Menrva en Titán, sobrevuelo T3](https://raw.githubusercontent.com/charlyacha/Labo1-colabs/revisados/Archivos_salida/06bis_AjusteNoLineal/radar_titan_cassini.png)

*Fuente: NASA/JPL — Cassini, sobrevuelo T3 (2005-02-15), imagen de dominio público.*

In [ ]:
# Una excelente costumbre es importar todo lo que necesitamos al principio del notebook, 
# así no nos olvidamos de nada y podemos ejecutar celdas en cualquier orden sin problemas.
# Tanto en este lenguaje como en cualquier otro.
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

np.random.seed(20260916) # semilla para reproducibilidad, igual que los mundo de minecraft

In [ ]:
# Supongamos que tengo una imagen satelital de un radar de apertura sintética (en microondas) 
# y quiero ajustar la dsitribución del ruido del radar:

##### Esto pueden ignorarlo, es solo para generar datos de ejemplo #####
# Generar una muestra aleatoria con distribución gamma
forma_real = 3.0
escala_real = 1.2
muestra = np.random.gamma(shape=forma_real, scale=escala_real, size=5000)
########################################################################


## Estos son sus datos eje X y eje Y:
#frecunecias juega el papel de eje Y, x_datos el de eje X
frecuencias, bordes = np.histogram(muestra, bins=20) 
x_datos = (bordes[:-1] + bordes[1:]) / 2

err_frec = frecuencias*0.03

In [ ]:
# Miremos que pinta tienen:

#Yo acá lo miro con plt.bar porque es una distribución, 
# si fuese un grafico y(x) puedo usar plt.plot o plt.scatter()
plt.bar(x_datos, frecuencias, width=bordes[1] - bordes[0], color = 'darkgreen', alpha=0.7, label="Muestra", zorder = 10)

# Esto son los puntos que quiero ajustar con sus incertezas en Y
plt.errorbar(x_datos, frecuencias, yerr = err_frec, color = 'k', fmt = 'o', 
             ecolor='grey', capsize=2, zorder = 5,
             label = 'Puntos a ajustar')

plt.title('Distribución de ruido Speckle de radar')
plt.xlabel('Potencia (mW)')
plt.ylabel('Conteo')
plt.grid(True, zorder = 0, lw = 0.5)
plt.legend()

In [ ]:
# Definamos ahora el modelo con el cual quiero ajustar los datos observados

# No importa que es, lo importante es que ustedes tienen que definir la función que quieren fitear
def funcion_gamma(x, amplitud, forma, escala):
    return amplitud * stats.gamma.pdf(x, a=forma, scale=escala)

In [ ]:
# Hagamos el ajuste de los datos con el modelo
parametros, pcov = curve_fit(
    funcion_gamma, # funcion/modelo que quiero ajustar
    x_datos, # datos en eje x
    frecuencias, # datos en eje y
    sigma = err_frec, # errores en el eje Y
    absolute_sigma = True, #Indicamos que el error es absoluto y no relativo, sinó va a forzar chi = 1
    p0=[muestra.size * (10 / 19), 2.0, 1.0], # "patada inicial", por donde empieza a buscar
    bounds=([0, 0.01, 0.01], [np.inf, np.inf, np.inf]) # Limites en donde se pueden mover cada parámetro
)

# Comentario: el dato de p0 es útil si sabemos mas o menos por donde debería buscar para minimizar 
# la función de costo (cuadrados minimos ponderados). Si no tenemos idea de que valores deberían ser
# podemos eliminarlo y que busque por su cuenta. Aunque tarda mas en conveger.

# Incertezas estimadas a partir de la matrix de covarianza:

delta_amplitud, delta_forma, delta_escala = np.sqrt(np.diag(pcov))

In [ ]:
from IPython.display import display, Math

# Defino un eje X en donde voy a evaluar la función y el eje Y como la función 
# evaluada con los parámetros encontrados
x_ajuste = np.linspace(min(x_datos), max(x_datos), 100)
y_ajuste = funcion_gamma(x_ajuste, *parametros) # el "*" sería como un "todo lo demás"

# Mostrar los datos y el ajuste
plt.errorbar(x_datos, frecuencias, yerr = err_frec, color = 'k', fmt = 'o', 
             ecolor='grey', capsize=2,
             label = 'Puntos a ajustar')

# Grafico la funcion 
plt.plot(x_ajuste, y_ajuste, "r-", label="Ajuste gamma")

plt.title('Distribución de ruido Speckle de radar')
plt.xlabel('Potencia (mW)')
plt.ylabel('Conteo')
plt.grid(True, zorder = 0, lw = 0.5)
plt.legend()

display(Math(rf"\mathrm{{Forma\ ajustada}} = {parametros[1]:.2f} \pm {delta_forma:.2f}"))
display(Math(rf"\mathrm{{Escala\ ajustada}} = {parametros[2]:.2f} \pm {delta_escala:.2f}"))
display(Math(rf"\mathrm{{Amplitud\ ajustada}} = {parametros[0]:.2f} \pm {delta_amplitud:.2f}"))

In [ ]:
# Veamos el gráfico de residuos:

residuos = frecuencias - funcion_gamma(x_datos, *parametros)

plt.figure(figsize=(8, 4))
plt.errorbar(
    x_datos, residuos, yerr=err_frec,
    fmt='o', color='k', ecolor='grey', capsize=2
)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Potencia (mW)')
plt.ylabel('Residuos')
plt.title('Residuos del ajuste gamma')
plt.grid(True, alpha=0.4)
plt.show()

In [ ]:
# Por ultimo veamos como da el chi-square

def chi2_reducido(y, y_modelo, sigma, n_parametros, verbose=True):
    """χ² reducido y su p-valor. sigma: incertezas experimentales reales."""
    y, y_modelo, sigma = map(lambda v: np.asarray(v, float), (y, y_modelo, sigma))
    chi2 = np.sum(((y - y_modelo) / sigma)**2)
    nu = len(y) - n_parametros
    chi2r = chi2 / nu
    p = stats.chi2.sf(chi2, nu)      # probabilidad de obtener un χ² igual o peor
    if verbose:
        if chi2r > 2:      lect = "modelo inadecuado o incertezas subestimadas"
        elif chi2r < 0.5:  lect = "incertezas probablemente sobreestimadas"
        else:              lect = "consistente con un buen ajuste"
        display(Math(
            rf"\chi^2 = {chi2:.2f}"
            rf"\quad \nu = {nu}"
            rf"\quad \chi_\nu^2 = {chi2r:.2f}"
            rf"\quad p = {p:.3f}"
        ))
        print(f"  -> {lect}")
    return chi2r, p

chi2_reducido(frecuencias, funcion_gamma(x_datos, *parametros), err_frec, n_parametros=3)